# Plot a turbine feature over time

Run notebook 02_parquet_setup/01_setup_yearly_parquets.ipynb then notebook 03_preprocessing/01_preprocess_10min_parquets.ipynb to prepare the 10-minute Parquet data. Set YEAR, TURBINE_ID, FEATURE and the date range below, then run all cells. Separate duration cells and alarm overlays are available further down. Reads Data_Parquets_10min; no Excel conversion is performed here.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

WORKSPACE_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'notebooks' / '01_data_export/01_export_yearly_turbine_scada.ipynb').is_file()),
    None,
)
if WORKSPACE_ROOT is None:
    raise FileNotFoundError('Open this notebook from Iteration2 or its notebooks folder.')


In [ ]:
# EDIT THESE SETTINGS, then rerun the cells below.
YEAR = 2016
TURBINE_ID = '2304511'
FEATURE = 'wtc_GearPres_mean'  # Ambient temperature; choose another name from the list below.
Y_LABEL = None  # Optional: a descriptive label with verified units.
START_DATE = f'{YEAR}-01-01'
END_DATE = f'{YEAR + 1}-01-01'  # Exclusive upper bound.

PARQUET_ROOT = WORKSPACE_ROOT / "Data_Parquets_10min"


In [ ]:
import sys
import json
if str(WORKSPACE_ROOT / 'notebooks' / 'shared') not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT / 'notebooks' / 'shared'))
from interval_parquet import load_aligned
frame, _ = load_aligned(PARQUET_ROOT, YEAR, TURBINE_ID)
_data = frame.reset_index().rename(columns={'alarmEvents': 'AlarmEvents'})
if frame.attrs.get('carry_in_alarm_events') and not _data.empty:
    _data.loc[0, 'AlarmEvents'] = json.dumps(json.loads(_data.loc[0, 'AlarmEvents']) + frame.attrs['carry_in_alarm_events'])
FEATURES = [c for c in _data if c.startswith('wtc_')]
print('Available features:', ', '.join(FEATURES))
if FEATURE not in FEATURES:
    raise ValueError(f'Unknown feature: {FEATURE}')


In [ ]:
# Exclude rows containing only alarm events; retain missing sensor values as gaps.
sensor_rows = _data[FEATURES].notna().any(axis=1)
series = _data.loc[sensor_rows, ['TimeStamp', FEATURE]].copy()
series['TimeStamp'] = pd.to_datetime(series['TimeStamp'], errors='coerce')
series[FEATURE] = pd.to_numeric(series[FEATURE], errors='coerce')
start, end = pd.Timestamp(START_DATE), pd.Timestamp(END_DATE)
if end <= start:
    raise ValueError('END_DATE must be after START_DATE.')
series = series.loc[
    series['TimeStamp'].ge(start) & series['TimeStamp'].lt(end)
].sort_values('TimeStamp')
if not series[FEATURE].notna().any():
    raise ValueError('No numeric readings for this feature and date range. Choose another feature or range.')
print(f'Turbine {TURBINE_ID}: {series[FEATURE].notna().sum():,} numeric readings; '
      f'{series[FEATURE].isna().sum():,} missing values in sensor rows.')
display(series.head())

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(series['TimeStamp'], series[FEATURE], linewidth=0.6, color='tab:blue')
ax.set(title=f'Turbine {TURBINE_ID} | {FEATURE} | {YEAR}',
       xlabel='Time', ylabel=Y_LABEL or FEATURE, xlim=(start, end))
locator = mdates.AutoDateLocator()
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()


## Feature plot with alarms
Run the new cell after the cells above. Red shading shows recorded alarm durations; dashed red lines mark starts with no usable duration. Edit `ALARM_CODES` to filter codes. These are turbine alarms, not necessarily faults related to the selected sensor.


In [ ]:
# Run after the feature plot above. Change this to select particular alarm codes.
# Example: ALARM_CODES = {'123', '456'}; None includes every recorded code.
import json
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

ALARM_CODES = None

# Read from the complete turbine data, including alarm-only rows.
# A blank Alarmcode on a sensor row does not mean that no alarm was active.
if 'AlarmEvents' not in _data.columns:
    raise ValueError('This workbook needs the AlarmEvents column from the exporter.')
selected_codes = None if ALARM_CODES is None else {str(code) for code in ALARM_CODES}
intervals, start_markers = [], []
event_count = 0
for payload in _data['AlarmEvents'].dropna():
    if not str(payload).strip():
        continue
    for event in json.loads(payload):
        if selected_codes is not None and str(event['Alarmcode']) not in selected_codes:
            continue
        alarm_start = pd.to_datetime(event.get('TimeOn'), errors='coerce')
        alarm_end = pd.to_datetime(event.get('TimeOff'), errors='coerce')
        if pd.isna(alarm_start):
            continue
        if pd.notna(alarm_end) and alarm_end > alarm_start:
            if alarm_start < end and alarm_end > start:
                intervals.append((max(alarm_start, start), min(alarm_end, end)))
                event_count += 1
        elif start <= alarm_start < end:
            # Unknown, zero or invalid duration: mark the start only.
            start_markers.append(alarm_start)
            event_count += 1

# Merge overlapping periods so frequent alarms do not make the shading darker.
merged_intervals = []
for left, right in sorted(intervals):
    if merged_intervals and left <= merged_intervals[-1][1]:
        merged_intervals[-1] = (merged_intervals[-1][0], max(right, merged_intervals[-1][1]))
    else:
        merged_intervals.append((left, right))

fig_alarm, ax_alarm = plt.subplots(figsize=(15, 5))
feature_line, = ax_alarm.plot(series['TimeStamp'], series[FEATURE],
                              linewidth=0.6, color='tab:blue', label=FEATURE)
for left, right in merged_intervals:
    ax_alarm.axvspan(left, right, color='tab:red', alpha=0.18, linewidth=0)
for alarm_start in sorted(set(start_markers)):
    ax_alarm.axvline(alarm_start, color='tab:red', alpha=0.45, linewidth=0.6, linestyle='--')
handles = [feature_line]
if merged_intervals:
    handles.append(Patch(facecolor='tab:red', alpha=0.18, label='Recorded alarm active period'))
if start_markers:
    handles.append(Line2D([0], [0], color='tab:red', linestyle='--',
                          label='Alarm start (duration unavailable or nonpositive)'))
ax_alarm.legend(handles=handles, loc='upper right')
ax_alarm.set(title=f'Turbine {TURBINE_ID} | {FEATURE} | {YEAR} | Alarm overlay',
             xlabel='Time', ylabel=Y_LABEL or FEATURE, xlim=(start, end))
locator = mdates.AutoDateLocator()
ax_alarm.xaxis.set_major_locator(locator)
ax_alarm.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
ax_alarm.grid(True, alpha=0.25)
fig_alarm.tight_layout()
print(f'{event_count:,} matching alarm events overlap the selected range or start within it.')
print('Red shading: recorded TimeOn to TimeOff. Dashed lines: starts without a usable duration.')
print('Only alarms present in this workbook can be shown; events starting outside the exported year may be absent.')
plt.show()


## Day, week, month and year views
Run the imports, settings, data-loading and helper cells once. Then run any view below independently. Each uses the selected `TURBINE_ID` and `FEATURE`, includes alarm highlights, and reuses loaded data. Edit the start date in the view you want. A month/year uses a calendar duration; the end is exclusive. Only data present in the loaded workbook can be shown.


In [ ]:
# Shared plotting helper: run this once before using the duration cells.
def plot_time_window(window_start, duration, alarm_codes=None):
    if '_data' not in globals():
        raise RuntimeError('Run the import, settings and data-loading cells above first, or select Run All.')
    start = pd.Timestamp(window_start)
    # Calendar offsets preserve real month lengths and leap years.
    length = duration if isinstance(duration, pd.DateOffset) else pd.Timedelta(duration)
    end = start + length
    if pd.isna(start) or pd.isna(end) or end <= start:
        raise ValueError('Choose a valid start date and a positive duration.')
    # Build the window from all loaded readings, independently of START_DATE/END_DATE.
    sensor_rows = _data[FEATURES].notna().any(axis=1)
    series = _data.loc[sensor_rows, ['TimeStamp', FEATURE]].copy()
    series['TimeStamp'] = pd.to_datetime(series['TimeStamp'], errors='coerce')
    series[FEATURE] = pd.to_numeric(series[FEATURE], errors='coerce')
    series = series.loc[series['TimeStamp'].ge(start) & series['TimeStamp'].lt(end)].sort_values('TimeStamp')
    if not series[FEATURE].notna().any():
        raise ValueError('No numeric readings in this window. Change WINDOW_START or DURATION; only the loaded workbook is available.')
    print(f'Displaying {start} to {end} (end exclusive).')
    # Run after the feature plot above. Change this to select particular alarm codes.
    # Example: ALARM_CODES = {'123', '456'}; None includes every recorded code.
    import json
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D

    ALARM_CODES = alarm_codes

    # Read from the complete turbine data, including alarm-only rows.
    # A blank Alarmcode on a sensor row does not mean that no alarm was active.
    if 'AlarmEvents' not in _data.columns:
        raise ValueError('This workbook needs the AlarmEvents column from the exporter.')
    selected_codes = None if ALARM_CODES is None else {str(code) for code in ALARM_CODES}
    intervals, start_markers = [], []
    event_count = 0
    for payload in _data['AlarmEvents'].dropna():
        if not str(payload).strip():
            continue
        for event in json.loads(payload):
            if selected_codes is not None and str(event['Alarmcode']) not in selected_codes:
                continue
            alarm_start = pd.to_datetime(event.get('TimeOn'), errors='coerce')
            alarm_end = pd.to_datetime(event.get('TimeOff'), errors='coerce')
            if pd.isna(alarm_start):
                continue
            if pd.notna(alarm_end) and alarm_end > alarm_start:
                if alarm_start < end and alarm_end > start:
                    intervals.append((max(alarm_start, start), min(alarm_end, end)))
                    event_count += 1
            elif start <= alarm_start < end:
                # Unknown, zero or invalid duration: mark the start only.
                start_markers.append(alarm_start)
                event_count += 1

    # Merge overlapping periods so frequent alarms do not make the shading darker.
    merged_intervals = []
    for left, right in sorted(intervals):
        if merged_intervals and left <= merged_intervals[-1][1]:
            merged_intervals[-1] = (merged_intervals[-1][0], max(right, merged_intervals[-1][1]))
        else:
            merged_intervals.append((left, right))

    fig_alarm, ax_alarm = plt.subplots(figsize=(15, 5))
    feature_line, = ax_alarm.plot(series['TimeStamp'], series[FEATURE],
                                  linewidth=0.6, color='tab:blue', label=FEATURE)
    for left, right in merged_intervals:
        ax_alarm.axvspan(left, right, color='tab:red', alpha=0.18, linewidth=0)
    for alarm_start in sorted(set(start_markers)):
        ax_alarm.axvline(alarm_start, color='tab:red', alpha=0.45, linewidth=0.6, linestyle='--')
    handles = [feature_line]
    if merged_intervals:
        handles.append(Patch(facecolor='tab:red', alpha=0.18, label='Recorded alarm active period'))
    if start_markers:
        handles.append(Line2D([0], [0], color='tab:red', linestyle='--',
                              label='Alarm start (duration unavailable or nonpositive)'))
    ax_alarm.legend(handles=handles, loc='upper right')
    ax_alarm.set(title=f'Turbine {TURBINE_ID} | {FEATURE} | {YEAR} | Alarm overlay',
                 xlabel='Time', ylabel=Y_LABEL or FEATURE, xlim=(start, end))
    locator = mdates.AutoDateLocator()
    ax_alarm.xaxis.set_major_locator(locator)
    ax_alarm.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax_alarm.grid(True, alpha=0.25)
    fig_alarm.tight_layout()
    print(f'{event_count:,} matching alarm events overlap the selected range or start within it.')
    print('Red shading: recorded TimeOn to TimeOff. Dashed lines: starts without a usable duration.')
    print('Only alarms present in this workbook can be shown; events starting outside the exported year may be absent.')
    plt.show()


### One day


In [ ]:
# Change this start date and rerun this cell.
if 'plot_time_window' not in globals() or '_data' not in globals():
    raise RuntimeError('Run the import, settings, data-loading and shared plotting helper cells first.')
DAY_START = f'{YEAR}-01-01'
DAY_ALARM_CODES = None  # All alarms; or a set of codes, e.g. {'123', '456'}.
plot_time_window(DAY_START, pd.DateOffset(days=1), DAY_ALARM_CODES)


### One week


In [ ]:
# Change this start date and rerun this cell.
if 'plot_time_window' not in globals() or '_data' not in globals():
    raise RuntimeError('Run the import, settings, data-loading and shared plotting helper cells first.')
WEEK_START = f'{YEAR}-01-01'
WEEK_ALARM_CODES = None  # All alarms; or a set of codes, e.g. {'123', '456'}.
plot_time_window(WEEK_START, pd.DateOffset(weeks=1), WEEK_ALARM_CODES)


### One month


In [ ]:
# Change this start date and rerun this cell.
if 'plot_time_window' not in globals() or '_data' not in globals():
    raise RuntimeError('Run the import, settings, data-loading and shared plotting helper cells first.')
MONTH_START = f'{YEAR}-01-01'
MONTH_ALARM_CODES = None  # All alarms; or a set of codes, e.g. {'123', '456'}.
plot_time_window(MONTH_START, pd.DateOffset(months=1), MONTH_ALARM_CODES)


### One year


In [ ]:
# Change this start date and rerun this cell.
if 'plot_time_window' not in globals() or '_data' not in globals():
    raise RuntimeError('Run the import, settings, data-loading and shared plotting helper cells first.')
YEAR_START = f'{YEAR}-01-01'
YEAR_ALARM_CODES = None  # All alarms; or a set of codes, e.g. {'123', '456'}.
plot_time_window(YEAR_START, pd.DateOffset(years=1), YEAR_ALARM_CODES)
